Red convolucional

In [11]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
from torch import nn, optim
import kagglehub
import os
import pandas as pd
from PIL import Image
import torch.nn.functional as F

In [ ]:
import os
import torch
import kagglehub
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms


# --- 1. Configuración y Descarga de Kaggle ---
KAGGLE_DATASET_ID = 'pkdarabi/cardetection'
print(f"Descargando dataset: {KAGGLE_DATASET_ID}...")
KAGGLE_DOWNLOAD_PATH = kagglehub.dataset_download(KAGGLE_DATASET_ID)
print(f"Dataset descargado en: {KAGGLE_DOWNLOAD_PATH}")


# --- 2. Rutas y Constantes ---
IMAGE_SIZE = (128, 128) # Reducido para que el cálculo del 'flatten' sea el del modelo
ORIGINAL_CSV_PATH = 'yolo_labels_dataset.csv' 
IMGS_DIR = os.path.join(KAGGLE_DOWNLOAD_PATH, 'car', 'train', 'images')

# Hiperparámetros del modelo
NUM_CLASSES = 14 # 15 clases originales, menos la clase 2 que eliminamos
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001

# Determinar el dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")


# --- 3. PROCESAMIENTO AUTOMÁTICO DEL CSV ---
print(f"\nProcesando el archivo CSV: {ORIGINAL_CSV_PATH}")
df_procesado = None
try:
    df_original = pd.read_csv(ORIGINAL_CSV_PATH)
    print(f"Filas originales: {len(df_original)}")
    print(f"Clases originales: {sorted(df_original['clase_indice'].unique())}")

    # Regla 1: Eliminar la clase 2
    df_procesado = df_original[df_original['clase_indice'] != 2].copy()
    
    # Regla 2: Re-mapear clases > 2
    df_procesado.loc[df_procesado['clase_indice'] > 2, 'clase_indice'] -= 1
    
    print(f"Filas después del procesado: {len(df_procesado)}")
    print(f"Clases re-mapeadas: {sorted(df_procesado['clase_indice'].unique())}")
    print("Procesamiento del CSV completado.")

except FileNotFoundError:
    print(f"Error: No se encontró el archivo '{ORIGINAL_CSV_PATH}'.")
    exit()
except Exception as e:
    print(f"Error procesando el CSV: {e}")
    exit()


# --- 4. Transformaciones ---
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE), # Asegúrate que coincida con el cálculo del modelo
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Normalizar
])


# --- 5. Clase YOLODataset (Modificada para aceptar un DataFrame) ---
class YOLODataset(Dataset):
    def __init__(self, labels_df, directorio_imagenes, transform=None):
        self.full_labels_df = labels_df
        self.directorio_imagenes = directorio_imagenes
        self.transform = transform
        self.imagenes_unicas = self.full_labels_df['nombre_archivo'].unique()
        self.labels_grouped = self.full_labels_df.groupby('nombre_archivo')

    def __len__(self):
        return len(self.imagenes_unicas)

    def __getitem__(self, idx):
        image_name = self.imagenes_unicas[idx]
        image_path = os.path.join(self.directorio_imagenes, image_name)
        
        try:
            image = Image.open(image_path).convert('RGB')
        except FileNotFoundError:
            print(f"Advertencia: No se encontró la imagen {image_path}.")
            # Devolver algo para que no se rompa el lote
            return torch.zeros(3, IMAGE_SIZE[0], IMAGE_SIZE[1]), torch.tensor(-1) 

        boxes_df = self.labels_grouped.get_group(image_name)
        
        # Tomar la primera etiqueta (modo Clasificación)
        clase_idx = int(boxes_df['clase_indice'].iloc[0])
        label = torch.tensor(clase_idx, dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label


# --- 6. Modelo CNN de PyTorch ---
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(SimpleCNN, self).__init__()
        
        # Input (3, 128, 128)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3) 
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # -> (32, 63, 63)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
        # -> (64, 30, 30)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3)
        # -> (128, 14, 14)
        
        # Tamaño aplanado = 128 (filtros) * 14 * 14 = 25088
        self.fc1 = nn.Linear(128 * 14 * 14, 128)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(128, num_classes) # Salida a 14 clases

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        
        x = x.view(x.size(0), -1) # Aplanar
        
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x) # Logits (sin softmax)
        return x


# --- 7. Crear Datasets y DataLoaders ---
if df_procesado is not None and not df_procesado.empty:
    
    
    # Crear DataFrames separados para train y val
    print(f"\nImágenes de entrenamiento: {len(df_train['nombre_archivo'].unique())}")
    print(f"Imágenes de validación: {len(df_val['nombre_archivo'].unique())}")

    # Crear instancias del Dataset
    train_dataset = YOLODataset(labels_df=df_train, directorio_imagenes=IMGS_DIR, transform=transform)
    val_dataset = YOLODataset(labels_df=df_val, directorio_imagenes=IMGS_DIR, transform=transform)

    # Crear los DataLoaders
    train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    print("DataLoaders creados.")


# --- 8. Inicializar Modelo, Pérdida y Optimizador ---
model = SimpleCNN(num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss() # Incluye Softmax
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("\nModelo, Pérdida y Optimizador inicializados.")
print(model)


# --- 9. Bucle de Entrenamiento y Validación ---
print(f"\n--- Iniciando Entrenamiento por {EPOCHS} epochs ---")

for epoch in range(EPOCHS):
    
    # --- Fase de Entrenamiento ---
    model.train() # Poner el modelo en modo entrenamiento
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for i, (images, labels) in enumerate(train_loader):
        # Omitir lotes con etiquetas -1 (imágenes no encontradas)
        if -1 in labels:
            continue
            
        images, labels = images.to(device), labels.to(device)
        
        # 1. Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # 2. Backward pass y optimización
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Calcular estadísticas
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct_train / total_train

    # --- Fase de Validación ---
    model.eval() # Poner el modelo en modo evaluación
    running_loss_val = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad(): # No calcular gradientes en validación
        for images, labels in val_loader:
            if -1 in labels:
                continue
                
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss_val += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_loss = running_loss_val / len(val_loader)
    val_acc = 100 * correct_val / total_val

    print(f"Epoch [{epoch+1}/{EPOCHS}] - "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% - "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

print("\n--- Entrenamiento Completado ---")

# Opcional: Guardar el modelo entrenado
torch.save(model.state_dict(), 'yolo_classifier_model.pth')
print("Modelo guardado en 'yolo_classifier_model.pth'")

Descargando dataset: pkdarabi/cardetection...
Dataset descargado en: C:\Users\ivanp\.cache\kagglehub\datasets\pkdarabi\cardetection\versions\5
Usando dispositivo: cpu

Procesando el archivo CSV: yolo_labels_dataset.csv
Filas originales: 4279
Clases originales: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13)]
Filas después del procesado: 4012
Clases re-mapeadas: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]
Procesamiento del CSV completado.


NameError: name 'train_test_split' is not defined

In [19]:
# --- Configuración del Loader ---
BATCH_SIZE = 16 # Tamaño de batch comúnmente usado en detección de objetos.

# Crear el DataLoader
train_loader = DataLoader(dataset_yolo, batch_size=BATCH_SIZE, shuffle=True)
print(f"\nTipo de DataLoader creado: {type(train_loader)}")

print(f"\nDataLoader creado con éxito. Número de batches: {len(train_loader)}")


Tipo de DataLoader creado: <class 'torch.utils.data.dataloader.DataLoader'>

DataLoader creado con éxito. Número de batches: 220


In [ ]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        # Primera capa convolutiva
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)  # Max pooling de 2x2
        # Segunda capa convolutiva
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        # Capa completamente conectada
        self.fc1 = nn.Linear(64 * 104 * 104, 120)  # Ajustamos correctamente las dimensiones
        self.fc2 = nn.Linear(120, 15)  # 15 categorías de salida

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # Conv1 -> ReLU -> Pooling
        x = self.pool(F.relu(self.conv2(x)))  # Conv2 -> ReLU -> Pooling
        x = x.view(-1, 64 * 104 * 104)  # Aplanamos el tensor
        x = F.relu(self.fc1(x))  # FC1 -> ReLU
        x = self.fc2(x)  # FC2
        return x

In [ ]:
# Instanciamos la red y configuramos el entrenamiento
model = ConvNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Entrenamiento
epochs = 40
for epoch in range(epochs):
    running_loss = 0.0
    for images, labels in train_loader:  # Asegúrate de que `train_loader` esté correctamente definido
        optimizer.zero_grad()  # Limpiamos los gradientes
        outputs = model(images)  # Pasamos las imágenes por la red
        loss = criterion(outputs, labels)  # Calculamos la pérdida
        loss.backward()  # Backpropagation
        optimizer.step()  # Actualizamos los pesos

        running_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {running_loss / len(train_loader)}')

Epoch 1, Loss: 2.476525477154762
Epoch 2, Loss: 1.6958120664859788
Epoch 3, Loss: 1.199205797452193
Epoch 4, Loss: 0.6972223429927998
Epoch 5, Loss: 0.3850776702249886
Epoch 6, Loss: 0.25472473877806606
Epoch 7, Loss: 0.1893362277967021
Epoch 8, Loss: 0.15575871300752953
Epoch 9, Loss: 0.1024858111281757
Epoch 10, Loss: 0.1063805805657236
Epoch 11, Loss: 0.09008203545133309
Epoch 12, Loss: 0.09436877746012787
Epoch 13, Loss: 0.11938132171474852
Epoch 14, Loss: 0.1407146173844333
Epoch 15, Loss: 0.06329090129273282
Epoch 16, Loss: 0.05410936498912831
Epoch 17, Loss: 0.060985499384815116
Epoch 18, Loss: 0.039992840466515175
Epoch 19, Loss: 0.04479353127358308
Epoch 20, Loss: 0.04404553123906506
Epoch 21, Loss: 0.05108949251797438
Epoch 22, Loss: 0.0967979999920535
Epoch 23, Loss: 0.1412093515525787
Epoch 24, Loss: 0.07469683110622301
Epoch 25, Loss: 0.06790913786402376
Epoch 26, Loss: 0.029649551007263224
Epoch 27, Loss: 0.03618424798806943
Epoch 28, Loss: 0.028901871803462372
Epoch 29, 

In [ ]:
# --- Configuración de rutas para test ---
IMAGEN_DIR_TEST = os.path.join(KAGGLE_DOWNLOAD_PATH, 'car', 'test', 'images') 
ETIQUETAS_DIR_TEST = os.path.join(KAGGLE_DOWNLOAD_PATH, 'car', 'test', 'labels')
CSV_SALIDA_TEST = 'yolo_labels_test.csv'

# Definir las rutas usando la ruta de descarga de Kaggle
ruta_csv_test = os.path.join(DATA_DIR, CSV_SALIDA_TEST) # El CSV se creó en el directorio actual
# IMPORTANTE: Definir la ruta de imágenes APUNTANDO al subdirectorio 'train/images'
ruta_imgs_test = os.path.join(KAGGLE_DOWNLOAD_PATH, 'car', 'test', 'images') 

# Crear DataLoader de test
dataset_yolo_test = YOLODataset(archivo_csv=ruta_csv_test, directorio_imagenes=ruta_imgs_test, transform=transform)
test_loader = DataLoader(dataset_yolo_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nTest DataLoader creado con {len(test_loader)} batches")



Test DataLoader creado con 40 batches


In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Evaluación de la red
def evaluate(model, test_loader):
    model.eval()  # Poner el modelo en modo evaluación
    correct = 0
    total = test_loader.dataset.__len__()  # Total de muestras en el conjunto de test
    print(f'Total de muestras en el conjunto de test: {total}')
    with torch.no_grad():  # No calcular gradientes
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)  # Mover datos al dispositivo
            outputs = model(inputs)  # Forward pass
            _, predicted = torch.max(outputs.data, 1)  # Obtener las predicciones
            correct += (predicted == labels).sum().item()  # Actualizar el contador de aciertos
    accuracy = 100 * correct / total if total > 0 else 0
    print(f'Accuracy: {accuracy:.2f}%')

In [ ]:
evaluate(model, test_loader)

Total de muestras en el conjunto de test: 637
Accuracy: 51.65%
